# Week 5: Dynamic Mapping & Time-Machine Simulation

**Student Worksheet** — Fill in the code cells using AI assistance or your own code.

This week you will:
1. Call the CWA real-time rainfall API
2. Parse nested JSON into GeoDataFrame
3. Build interactive Folium maps with conditional styling
4. "Replay" Typhoon Fung-wong (2025) as a stress test
5. Overlay dynamic rainfall with shelter risk data

**Packages needed:** `geopandas`, `folium`, `requests`, `python-dotenv`, `branca`

## Cell [1]: Setup & Load Shelter Data

**What to do:**
- Import all required packages
- Load environment variables from .env
- Load shelter data from Week 3-4 (or create synthetic data)
- Print data summary

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
import requests
import json
import os
from dotenv import load_dotenv
from shapely.geometry import Point

# 2. Load .env
load_dotenv()

# 3. Load Real Shelter Data from CSV
shelter_csv_path = '/content/drive/MyDrive/GIS_data/cleaned_shelter_data.csv'
if os.path.exists(shelter_csv_path):
    df_real_shelters = pd.read_csv(shelter_csv_path)
    print("--- CSV Columns found ---")
    print(df_real_shelters.columns.tolist())

    # Manual mapping for coordinates
    lon_col = next((c for c in df_real_shelters.columns if c.lower() in ['lon', 'longitude', 'x', '經度']), None)
    lat_col = next((c for c in df_real_shelters.columns if c.lower() in ['lat', 'latitude', 'y', '緯度']), None)

    if lon_col and lat_col:
        # FIX: The CSV data is already in Lat/Lon degrees, so we set CRS to EPSG:4326
        gdf_shelters = gpd.GeoDataFrame(
            df_real_shelters,
            geometry=gpd.points_from_xy(df_real_shelters[lon_col], df_real_shelters[lat_col]),
            crs="EPSG:4326"
        )
        print(f"Successfully loaded {len(gdf_shelters)} shelters with WGS84 coordinates.")
    else:
        print("Error: Could not find coordinate columns.")
else:
    print(f"Error: File not found at {shelter_csv_path}")

if 'gdf_shelters' in locals():
    display(gdf_shelters.head())

## Cell [2]: Fetch CWA Rainfall API

**What to do:**
- Write a function `fetch_cwa_api(api_key)` that calls the CWA rainfall endpoint
- Handle errors gracefully
- Return JSON response

In [ ]:
def fetch_cwa_api(api_key):
    """
    Calls the CWA rainfall API endpoint O-A0002-001.
    Returns JSON response on success, None on failure.
    """
    url = "https://opendata.cwa.gov.tw/api/v1/rest/datastore/O-A0002-001"
    params = {
        "Authorization": api_key,
        "format": "JSON"
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status() # Check for HTTP errors
        return response.json()
    except Exception as e:
        print(f"Error fetching CWA API: {e}")
        return None

# Example usage (commented out until API_KEY is provided):
# api_key = os.getenv('CWA_API_KEY')
# data = fetch_cwa_api(api_key)
# if data: print('Successfully fetched data')

## Cell [3]: Parse Rainfall JSON → GeoDataFrame

**What to do:**
- Extract station data from nested JSON structure
- Filter out invalid data (-998 values)
- Create GeoDataFrame with proper CRS

In [ ]:
def normalize_cwa_json(raw):
    if not raw: return []
    stations = []
    if isinstance(raw, list): stations = raw
    elif isinstance(raw, dict):
        if 'records' in raw:
            recs = raw['records']
            stations = recs.get('Station', recs.get('location', []))
        elif 'cwaopendata' in raw:
            dataset = raw['cwaopendata'].get('dataset', {})
            stations = dataset.get('Station', dataset.get('location', []))
        elif 'Station' in raw: stations = raw['Station']
    if isinstance(stations, dict): return [stations]
    return stations if isinstance(stations, list) else []

def parse_rainfall_json(data):
    stations = normalize_cwa_json(data)
    parsed_list = []
    for s in stations:
        if not isinstance(s, dict): continue
        lat, lon = None, None

        # 1. Flexible Coordinate Extraction (Standard vs Simulation)
        coords = s.get('Coordinates', s.get('coordinates', []))
        if isinstance(coords, list) and len(coords) > 0:
            target = next((c for c in coords if isinstance(c, dict) and c.get('CoordinateName') == 'WGS84'), coords[0])
            if isinstance(target, dict):
                lat, lon = target.get('Latitude', target.get('StationLatitude')), target.get('Longitude', target.get('StationLongitude'))

        # Fallback for Simulation nested GeoInfo coordinates
        if (lat is None or lon is None) and isinstance(s.get('GeoInfo'), dict):
            gi = s['GeoInfo']
            gi_coords = gi.get('Coordinates', [])
            if isinstance(gi_coords, list) and len(gi_coords) > 0:
                lat = gi_coords[0].get('StationLatitude', gi_coords[0].get('Latitude'))
                lon = gi_coords[0].get('StationLongitude', gi_coords[0].get('Longitude'))

        if lat is None or lon is None: continue

        # 2. Precipitation Extraction
        rain_stats = s.get('RainfallElement', s.get('weatherElement', {}))
        def get_val(keys):
            curr = rain_stats
            for k in keys:
                if isinstance(curr, dict): curr = curr.get(k, {})
                elif isinstance(curr, list) and len(curr) > 0: curr = curr[0]
                else: return 0.0
            try:
                v = float(curr) if not isinstance(curr, (dict, list)) else 0.0
                return v if v >= 0 else 0.0
            except: return 0.0

        r1 = get_val(['Past1hr', 'Precipitation']) or get_val(['Now', 'Precipitation']) or get_val(['past1hr', 'precipitation']) or 0.0
        r3 = get_val(['Past3hr', 'Precipitation']) or 0.0
        r24 = get_val(['Past24hr', 'Precipitation']) or 0.0

        gi = s.get('GeoInfo', {})
        parsed_list.append({
            'StationName': s.get('StationName', s.get('locationName')),
            'StationId': s.get('StationId', s.get('stationId')),
            'CountyName': gi.get('CountyName') if isinstance(gi, dict) else None,
            'TownName': gi.get('TownName') if isinstance(gi, dict) else None,
            'rain_1hr': r1, 'rain_3hr': r3, 'rain_24hr': r24,
            'geometry': Point(float(lon), float(lat))
        })

    if not parsed_list:
        return gpd.GeoDataFrame(columns=['StationName', 'StationId', 'CountyName', 'TownName', 'rain_1hr', 'rain_3hr', 'rain_24hr', 'geometry'], geometry='geometry', crs="EPSG:4326")

    return gpd.GeoDataFrame(parsed_list, crs="EPSG:4326")

## Cell [4]: Mode Switcher (LIVE vs SIMULATION)

**What to do:**
- Read `APP_MODE` from .env
- If LIVE: fetch real rainfall data from API
- If SIMULATION: load fallback JSON file
- Parse using the same function in both cases

In [ ]:
import json
import os

path = os.getenv('SIMULATION_DATA')
if path and os.path.exists(path):
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    stations = []
    if 'records' in data and 'Station' in data['records']:
        stations = data['records']['Station']

    if stations and len(stations) > 0:
        print("--- First Station Object Detail ---")
        import pprint
        pprint.pprint(stations[0])
    else:
        print("No stations found in the expected 'records.Station' path.")
else:
    print("File not found.")

In [ ]:
import json
import os

path = os.getenv('SIMULATION_DATA')
if path and os.path.exists(path):
    with open(path, 'r', encoding='utf-8') as f:
        diag_data = json.load(f)

    print("--- JSON Root Keys ---")
    if isinstance(diag_data, dict):
        root_keys = list(diag_data.keys())
        print(root_keys)
        # Preview first level of nesting
        for k in root_keys[:3]:
            if isinstance(diag_data[k], dict):
                sub_keys = list(diag_data[k].keys())
                print(f"Keys under '{k}': {sub_keys[:5]}")
            elif isinstance(diag_data[k], list) and len(diag_data[k]) > 0:
                if isinstance(diag_data[k][0], dict):
                    print(f"List under '{k}' first element keys: {list(diag_data[k][0].keys())[:5]}")
                else:
                    print(f"List under '{k}' contains non-dict elements.")
    else:
        print(f"Data is a list of length {len(diag_data)}")
        if len(diag_data) > 0: print(f"First element type: {type(diag_data[0])}")
else:
    print("File path invalid or not found.")

## Cell [5]: Create Base Folium Map

**What to do:**
- Create a Folium map centered on Hualien County
- Use OpenStreetMap or Satellite basemap
- Set initial zoom level

In [ ]:
# Create a base Folium map centered on Hualien County
m = folium.Map(location=[23.98, 121.55], zoom_start=10, tiles='OpenStreetMap')

# Display the map
m

## Cell [6]: Add Rainfall CircleMarkers with Conditional Styling

**What to do:**
- Write a function `rain_color(rain_value)` that returns color based on rainfall amount
- Add CircleMarker for each rainfall station
- Size and color represent rainfall intensity

In [ ]:
def rain_color(rain_mm):
    if rain_mm < 10:
        return 'green'
    elif 10 <= rain_mm < 40:
        return 'gold'
    elif 40 <= rain_mm < 80:
        return 'orange'
    else:
        return 'red'

def rain_radius(rain_mm):
    return max(5, rain_mm / 5)

# Add CircleMarkers for each station in gdf_rainfall
for idx, row in gdf_rainfall.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=rain_radius(row['rain_1hr']),
        color=rain_color(row['rain_1hr']),
        fill=True,
        fill_color=rain_color(row['rain_1hr']),
        fill_opacity=0.6,
        tooltip=f"{row['StationName']}: {row['rain_1hr']} mm/hr"
    ).add_to(m)

m

## Cell [7]: Add HeatMap Layer

**What to do:**
- Import HeatMap from folium.plugins
- Create a heat layer showing rainfall intensity
- Add to folium map

In [ ]:
from folium.plugins import HeatMap

# Prepare data for HeatMap: list of [lat, lon, intensity]
heat_data = [[row.geometry.y, row.geometry.x, row['rain_1hr']] for idx, row in gdf_rainfall.iterrows() if row['rain_1hr'] > 0]

# Add HeatMap layer to map m
HeatMap(heat_data, name='Rainfall Heatmap', show=False).add_to(m)

print("Added Rainfall Heatmap layer to map.")

## Cell [8]: Add LayerControl

**What to do:**
- Enable layer visibility toggle for CircleMarkers and HeatMap
- Add LayerControl to map

In [ ]:
from folium import LayerControl

# Add LayerControl to the map to allow toggling layers
LayerControl(collapsed=False).add_to(m)

# Display the map
m

## Cell [9]: Add Shelter Risk Popups

**What to do:**
- Add shelter locations to map
- Color-code by risk_level
- Include rich popup with shelter name and risk info

In [ ]:
# Map risk levels to colors
risk_colors = {'low': 'blue', 'medium': 'orange', 'high': 'red'}

# Add shelter markers with rich popups
for idx, row in gdf_shelters.iterrows():
    # Convert CRS for Folium display
    lat_lon_geom = gdf_shelters.to_crs("EPSG:4326").iloc[idx].geometry

    # Use real column names from CSV
    shelter_name = row.get('避難收容處所名稱', '未知')
    address = row.get('避難收容處所地址', '無地址資料')
    capacity = row.get('預計收容人數', 0)

    popup_html = f"""
    <b>{shelter_name}</b><br>
    <b>地址:</b> {address}<br>
    <b>容量:</b> {capacity} 人
    """

    folium.Marker(
        location=[lat_lon_geom.y, lat_lon_geom.x],
        icon=folium.Icon(color='blue', icon='info-sign'),
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=shelter_name
    ).add_to(m)

m

---

# Lab 1: CWA API → Folium Map (25 minutes)

**Goal**: Call the rainfall API (or load fallback), parse JSON, create an interactive Folium map.

> **Fallback**: If CWA API doesn't work, load `data/scenarios/fungwong_202511.json` instead. The structure is similar (both use `records.Station[]`) but has minor differences — your `parse_rainfall_json()` should handle both via `normalize_cwa_json()`.

**Checklist:**
- [ ] Rainfall data loaded (API or fallback)
- [ ] GeoDataFrame parsed with correct CRS
- [ ] Folium map created with CircleMarkers
- [ ] Map saved as HTML
- [ ] Can toggle layers on/off

### Lab 1 Step 1: Load Data (API or Fallback)

In [28]:
from google.colab import userdata
import os
import json

# 1. 安全地從 Colab Secrets 讀取 API_KEY
try:
    api_key = userdata.get('CWA_API_KEY')
    print("成功從 Colab Secrets 讀取 API_KEY")
except:
    api_key = os.getenv('CWA_API_KEY')

# 2. 嘗試抓取即時資料
raw_data = fetch_cwa_api(api_key)

# 3. 如果 API 抓取失敗或沒有金鑰，則載入回溯模擬資料
if not raw_data:
    print("API 抓取失敗或金鑰缺失。正在載入備用模擬資料...")
    sim_path = os.getenv('SIMULATION_DATA', 'fungwong_202511.json')
    if os.path.exists(sim_path):
        with open(sim_path, 'r', encoding='utf-8') as f:
            raw_data = json.load(f)
    else:
        print(f"錯誤：找不到路徑 {sim_path} 的檔案")

# 4. 解析 JSON → gdf_rainfall
if raw_data:
    gdf_rainfall = parse_rainfall_json(raw_data)

    # 5. 輸出資訊
    print(f"資料筆數: {gdf_rainfall.shape[0]}")
    print(f"座標系統: {gdf_rainfall.crs}")
    display(gdf_rainfall.head())
else:
    print("未能成功載入任何資料。")

成功從 Colab Secrets 讀取 API_KEY
資料筆數: 1305
座標系統: EPSG:4326


,StationName,StationId,CountyName,TownName,rain_1hr,rain_3hr,rain_24hr,geometry
0,九份二山,C1I230,南投縣,國姓鄉,0.0,0.0,0.0,POINT (120.83686 23.96371)
1,基隆,466940,基隆市,仁愛區,0.0,0.0,0.5,POINT (121.73224 25.1351)
2,淡水,466900,新北市,淡水區,0.0,0.0,0.0,POINT (121.44067 25.16666)
3,新北,466881,新北市,新店區,0.0,0.0,0.0,POINT (121.51697 24.96099)
4,陽明山,466930,臺北市,北投區,0.0,0.0,0.0,POINT (121.53631 25.16386)


### Lab 1 Step 2: Parse JSON → GeoDataFrame

In [ ]:
# 1. Check columns and CRS
print(f"CRS check: {gdf_rainfall.crs}")
print(f"Expected columns present: {'rain_1hr' in gdf_rainfall.columns}")

# 2. Display first 5 rows
display(gdf_rainfall.head(5))

# 3. Print statistics for rainfall columns
print("\n--- Rainfall Statistics ---")
display(gdf_rainfall[['rain_1hr', 'rain_3hr', 'rain_24hr']].describe())

# 4. Find the highest rainfall station
max_rain_row = gdf_rainfall.loc[gdf_rainfall['rain_1hr'].idxmax()]
print(f"\nHighest rainfall station: {max_rain_row['StationName']} with {max_rain_row['rain_1hr']} mm/hr")

### Lab 1 Step 3: Build Folium Map + CircleMarkers

In [ ]:
# 1. Create base Folium map
m = folium.Map(location=[23.98, 121.55], zoom_start=10, tiles='OpenStreetMap')

# 2. Add CircleMarkers for rainfall stations
for idx, row in gdf_rainfall.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=rain_radius(row['rain_1hr']),
        color=rain_color(row['rain_1hr']),
        fill=True,
        fill_color=rain_color(row['rain_1hr']),
        fill_opacity=0.6,
        tooltip=f"{row['StationName']}: {row['rain_1hr']} mm/hr"
    ).add_to(m)

# 3. Add HeatMap layer
from folium.plugins import HeatMap
heat_data = [[row.geometry.y, row.geometry.x, row['rain_1hr']] for idx, row in gdf_rainfall.iterrows() if row['rain_1hr'] > 0]
HeatMap(heat_data, name='Rainfall Heatmap', show=False).add_to(m)

# 4. Add LayerControl
folium.LayerControl(collapsed=False).add_to(m)

# 5. Display map
m

### Lab 1 Step 4: Save Map as HTML

In [ ]:
import os

# 1. Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

# 2. Save map to 'output/rainfall_map_week5.html'
output_path = 'output/rainfall_map_week5.html'
m.save(output_path)

# 3. Verify file was created and print its size
if os.path.exists(output_path):
    file_size = os.path.getsize(output_path) / 1024
    print(f"Success! Map saved to {output_path}")
    print(f"File size: {file_size:.2f} KB")
else:
    print("Error: Map file was not created.")

---

# 🔬 Lab 2: Typhoon Fung-wong Simulation (15 minutes)

**Goal**: Switch to SIMULATION mode, overlay typhoon rainfall with shelter risk data.

> **Context**: It's 2025-11-11 14:00. Typhoon Fung-wong is hitting eastern Taiwan.
> Suao: 130.5mm/hr. Mataian Creek is forming a landslide dam.

**Checklist:**
- [x] Simulation data loaded
- [x] High-rainfall stations identified
- [x] Shelters within 5km radius found
- [x] Risk map created and saved

### Lab 2 Step 1: Load Simulation JSON + Filter High-Rain Stations

In [ ]:
# 1. Load 'fungwong_202511.json'
sim_path = os.getenv('SIMULATION_DATA', 'fungwong_202511.json')
with open(sim_path, 'r', encoding='utf-8') as f:
    sim_raw = json.load(f)

# 2. Parse simulation JSON
gdf_fungwong = parse_rainfall_json(sim_raw)

# 3. Filter: rain_1hr > 30 mm/hr (heavy rain)
heavy_rain_gdf = gdf_fungwong[gdf_fungwong['rain_1hr'] > 30].copy()

# 4. Print stats
print(f"Found {len(heavy_rain_gdf)} stations with rain > 30mm/hr.")

# 5. Find station with max rain_1hr
max_rain_idx = gdf_fungwong['rain_1hr'].idxmax()
max_rain_station = gdf_fungwong.loc[max_rain_idx]
print(f"Max Rainfall: {max_rain_station['StationName']} ({max_rain_station['rain_1hr']} mm/hr)")

display(heavy_rain_gdf.sort_values(by='rain_1hr', ascending=False).head())

### Lab 2 Step 2: Spatial Join Rainfall with Shelters

In [ ]:
# 1. Reproject rainfall to EPSG:3826 (meters) for buffering
gdf_rain_3826 = gdf_fungwong.to_crs("EPSG:3826")

# 2. Filter high-rain stations (> 40mm/hr)
high_rain = gdf_rain_3826[gdf_rain_3826['rain_1hr'] > 40].copy()

# 3. Create 5km (5000m) buffer
high_rain['geometry'] = high_rain.buffer(5000)

# 4. Spatial Join to find shelters within impact zones
# Note: gdf_shelters is already EPSG:3826 from Cell [1]
shelters_at_risk = gpd.sjoin(gdf_shelters, high_rain[['rain_1hr', 'geometry']], how='left', predicate='within')

# 5. Apply dynamic risk logic (Simplifying as real CSV lacks some terrain metrics)
def calculate_dynamic_risk(row):
    if pd.isna(row['rain_1hr']):
        return False, "NORMAL"

    if row['rain_1hr'] > 80:
        return True, "CRITICAL"
    else:
        return True, "WARNING"

# Apply the logic
risk_results = shelters_at_risk.apply(calculate_dynamic_risk, axis=1)
shelters_at_risk['is_at_risk'] = [r[0] for r in risk_results]
shelters_at_risk['dynamic_status'] = [r[1] for r in risk_results]

# 6. Display results using correct column names
risky_count = shelters_at_risk['is_at_risk'].sum()
print(f"Spatial Join complete. {risky_count} shelters are in high-rainfall zones.")
display(shelters_at_risk[shelters_at_risk['is_at_risk'] == True][['避難收容處所名稱', '縣市及鄉鎮市區', 'rain_1hr', 'dynamic_status']])

### Lab 2 Step 3: Final Map + Save HTML

In [ ]:
from folium.plugins import HeatMap, MarkerCluster
from branca.element import Element

# 1. 讀取環境變數決定模式
load_dotenv(override=True)
mode = os.getenv('APP_MODE', 'SIMULATION')
print(f"正在以 {mode} 模式產生最終風險地圖...")

# 2. 獲取對應數據
if mode == 'LIVE':
    raw_data = fetch_cwa_api(os.getenv('CWA_API_KEY'))
else:
    sim_path = os.getenv('SIMULATION_DATA', 'fungwong_202511.json')
    with open(sim_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)

gdf_final_rain = parse_rainfall_json(raw_data)

# 3. 建立地圖中心
target_gdf = shelters_at_risk if 'shelters_at_risk' in locals() else gdf_shelters

if not target_gdf.empty:
    gdf_plot = target_gdf.to_crs("EPSG:4326")
    center_lat = gdf_plot.geometry.y.mean()
    center_lon = gdf_plot.geometry.x.mean()
else:
    center_lat, center_lon = 23.98, 121.55

m_final = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='OpenStreetMap')

# --- 修正後的 CSS：僅針對叢集外圈，不影響內部 Marker ---
custom_css = """
<style>
.leaflet-marker-icon.marker-cluster { background-color: rgba(100, 100, 100, 0.5) !important; }
.marker-cluster div { background-color: rgba(150, 150, 150, 0.6) !important; border-radius: 20px; }
.marker-cluster span { color: white !important; font-weight: bold; }
</style>
"""
m_final.get_root().header.add_child(Element(custom_css))

# 4. 圖層：降雨測站標記
fg_stations = folium.FeatureGroup(name='降雨測站標記', show=True)
for _, row in gdf_final_rain.iterrows():
    if row['rain_1hr'] > 0:
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=rain_radius(row['rain_1hr']),
            color=rain_color(row['rain_1hr']),
            fill=True,
            fill_color=rain_color(row['rain_1hr']),
            fill_opacity=0.6,
            tooltip=f"{row['StationName']}: {row['rain_1hr']} mm/hr"
        ).add_to(fg_stations)
fg_stations.add_to(m_final)

# 5. 圖層：降雨熱度圖
heat_data = [[row.geometry.y, row.geometry.x, row['rain_1hr']] for _, row in gdf_final_rain.iterrows() if row['rain_1hr'] > 0]
fg_heat = folium.FeatureGroup(name='降雨熱度圖', show=False)
HeatMap(heat_data).add_to(fg_heat)
fg_heat.add_to(m_final)

# 6. 圖層：避難所 (叢集模式)
if not target_gdf.empty:
    fg_shelters = folium.FeatureGroup(name='避難所位置', show=True)
    marker_cluster = MarkerCluster().add_to(fg_shelters)

    for _, row in gdf_plot.iterrows():
        s_name = row.get('避難收容處所名稱', '未知名稱')
        at_risk = row.get('is_at_risk', False)
        status = row.get('dynamic_status', 'NORMAL')
        # 使用更鮮明的顏色：紅色(警告) 與 藍色(正常)
        color = 'red' if at_risk else 'blue'

        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            icon=folium.Icon(color=color, icon='home'),
            tooltip=s_name,
            popup=folium.Popup(f"<b>{s_name}</b><br>狀態: {status}", max_width=200)
        ).add_to(marker_cluster)
    fg_shelters.add_to(m_final)

folium.LayerControl(collapsed=False).add_to(m_final)
display(m_final)

---

# 💭 My Reflection (fill in your answers)

In [ ]:
from google.colab import files
import os

output_path = 'output/rainfall_map_week5.html'

# 檢查檔案是否存在
if os.path.exists(output_path):
    files.download(output_path)
    print(f"正在為您下載檔案: {output_path}")
else:
    print(f"錯誤：檔案 {output_path} 不存在。請確認已成功產生地圖。")

### 1. 從 LIVE 切換到 SIMULATION 模式時，你修改了幾行程式碼？

*你的回答：* 幾乎不需要修改。透過使用 `normalize_cwa_json` 函數和環境變數 `APP_MODE`，同一套解析邏輯就能同時處理即時 API 和本地模擬 JSON 檔案。

---

### 2. 如果在進行空間交集（sjoin）前忘記轉換 CRS 會發生什麼事？

*你的回答：* 空間交集會失敗或產生錯誤結果。因為原始座標是經緯度（EPSG:4326），而緩衝區距離單位是公尺（5000m）。若沒轉換，系統會把 5000 誤認為 5000 度，這足以覆蓋整個地球！

---

### 3. 為什麼 CWA 使用 -998 而不是 NaN 或 null？

*你的回答：* 歷史上許多科學數據格式（如 Fortran 或早期 CSV 系統）使用特定的數值代碼（如 -998 或 -999）來代表缺失資料或設備故障，這能確保欄位維持數值型態，方便舊型系統處理。

---

### 4. 在鳳凰颱風模擬期間，你會優先撤離哪一個避難所？為什麼？

*你的回答：* 雖然花蓮避難所未直接進入蘇澳強降雨（130mm/hr）的 5 公里範圍，但我會優先關注『秀林鄉避難所K』。因為它本身具備『中等』地形風險且海拔較高，一旦雨勢南移，最容易因道路中斷而受困。

---

### 5. 你在本次實驗中遇到什麼挑戰？如何解決？

*你的回答：* 最大的挑戰是 CWA JSON 的巢狀結構不一致，以及實作『即時/模擬』切換功能。寫了一個函數來處理不同的 Key 值，並利用 `ipywidgets` 製作切換鈕與『一鍵更新』邏輯，解決了切換設定後地圖不會自動更新的問題。完成以後嘗試帶入真實避難所資料實作，具實際庇護所資訊量太大，需一些處理才能流暢縮放。

以下是測試用課外嘗試區


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1. Create a radio button for mode selection
mode_widget = widgets.RadioButtons(
    options=['LIVE', 'SIMULATION'],
    value=os.getenv('APP_MODE', 'SIMULATION'),
    description='Data Mode:',
    disabled=False
)

output = widgets.Output()

def on_mode_change(change):
    with output:
        clear_output()
        new_mode = change['new']
        print(f"Switching to {new_mode} mode...")

        # Update .env file content
        with open('.env', 'r') as f:
            lines = f.readlines()

        with open('.env', 'w') as f:
            for line in lines:
                if line.startswith('APP_MODE='):
                    f.write(f"APP_MODE={new_mode}\n")
                else:
                    f.write(line)

        # Reload env and execute the loading cell logic
        load_dotenv(override=True)
        print(f"Successfully updated .env. Now please re-run Cell [4] and the Map cells.")

mode_widget.observe(on_mode_change, names='value')

print("--- Mode Switcher ---")
display(mode_widget, output)

In [ ]:
# 1. 重新載入環境變數
from folium.plugins import HeatMap, MarkerCluster
load_dotenv(override=True)
mode = os.getenv('APP_MODE', 'SIMULATION')
print(f'正在以 {mode} 模式重新產生數據與地圖...')

# 2. 獲取數據
if mode == 'LIVE':
    raw_data = fetch_cwa_api(os.getenv('CWA_API_KEY'))
else:
    sim_path = os.getenv('SIMULATION_DATA', 'fungwong_202511.json')
    with open(sim_path, 'r', encoding='utf-8') as f:
        raw_data = json.load(f)
gdf_current = parse_rainfall_json(raw_data)

# 3. 處理避難所座標轉換
if 'gdf_shelters' in locals() and not gdf_shelters.empty:
    # 強制轉換為經緯度 EPSG:4326
    gdf_s_4326 = gdf_shelters.to_crs("EPSG:4326")

    # 偵錯：列印前三筆轉換後的座標
    print("--- 座標轉換檢查 (前三筆) ---")
    for i in range(min(3, len(gdf_s_4326))):
        geom = gdf_s_4326.iloc[i].geometry
        print(f"避難所 {i}: Lat={geom.y:.4f}, Lon={geom.x:.4f}")

    # 計算地圖中心點
    center_lat = gdf_s_4326.geometry.y.mean()
    center_lon = gdf_s_4326.geometry.x.mean()
else:
    center_lat, center_lon = 23.98, 121.55
    print("警告：找不到避難所資料，使用預設中心點。")

# 4. 建立地圖
m_new = folium.Map(location=[center_lat, center_lon], zoom_start=8)

# --- 圖層：雨量熱度圖 ---
heat_data = [[row.geometry.y, row.geometry.x, row['rain_1hr']] for _, row in gdf_current.iterrows() if row['rain_1hr'] > 0]
fg_heat = folium.FeatureGroup(name='雨量熱度圖', show=True)
HeatMap(heat_data).add_to(fg_heat)
fg_heat.add_to(m_new)

# --- 圖層：避難所叢集 ---
if 'gdf_s_4326' in locals():
    fg_shelters = folium.FeatureGroup(name='避難所位置(叢集)', show=True)
    marker_cluster = MarkerCluster().add_to(fg_shelters)

    for _, row in gdf_s_4326.iterrows():
        s_name = row.get('避難收容處所名稱', '未知名稱')
        folium.Marker(
            location=[row.geometry.y, row.geometry.x],
            icon=folium.Icon(color='blue', icon='home'),
            tooltip=s_name,
            popup=f"<b>{s_name}</b>"
        ).add_to(marker_cluster)

    fg_shelters.add_to(m_new)
    print(f"地圖已載入 {len(gdf_s_4326)} 個避難所。")

folium.LayerControl(collapsed=False).add_to(m_new)
display(m_new)